In [1]:
import pandas as pd
import numpy as np
import random
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# 1. Re-load the raw data to ensure we have all columns (trip_id, stop_sequence, direction_id)
stops_df = pd.read_csv('../data/raw/stops.txt')
routes_df = pd.read_csv('../data/raw/routes.txt')
trips_df = pd.read_csv('../data/raw/trips.txt')
stop_times_df = pd.read_csv('../data/raw/stop_times.txt')

trip_route_df = pd.merge(trips_df, routes_df, on='route_id', how='left')
schedule_df = pd.merge(stop_times_df, trip_route_df, on='trip_id', how='left')
master_df = pd.merge(schedule_df, stops_df, on='stop_id', how='left')

# Filter Red Line
red_line_df = master_df[master_df['route_short_name'] == 'C1_RED'].copy()

# Task 4: Add Important Features (is_interchange_station)
interchanges = ['Ameerpet', 'MG Bus Station', 'Parade Ground']
red_line_df['is_interchange'] = red_line_df['stop_name'].apply(lambda x: 1 if x in interchanges else 0)

# Extract hour and basic time variables
def time_to_seconds(time_str):
    if pd.isna(time_str): return np.nan
    h, m, s = map(int, time_str.split(':'))
    return h * 3600 + m * 60 + s

red_line_df['scheduled_seconds'] = red_line_df['arrival_time'].apply(time_to_seconds)
red_line_df = red_line_df.dropna(subset=['scheduled_seconds']).copy()
red_line_df['hour'] = (red_line_df['scheduled_seconds'] // 3600) % 24
red_line_df['is_peak_hour'] = red_line_df['hour'].apply(lambda x: 1 if (8 <= x <= 10) or (17 <= x <= 20) else 0)
red_line_df['day_of_week'] = np.random.randint(0, 7, size=len(red_line_df))
red_line_df['is_weekend'] = red_line_df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# Task 5: Improve Synthetic Delay (Propagation & Trip Noise)
print("Generating Advanced Synthetic Delays...")
np.random.seed(42)

# Sort by trip and sequence to calculate propagation
red_line_df = red_line_df.sort_values(by=['trip_id', 'stop_sequence'])

delay_records = []
current_trip = None
cumulative_delay = 0

for index, row in red_line_df.iterrows():
    if row['trip_id'] != current_trip:
        # New trip starts, reset delay and add base noise
        current_trip = row['trip_id']
        cumulative_delay = np.random.exponential(scale=1.0) # Base trip delay
    
    # Add propagation: slight random increase at each subsequent stop
    cumulative_delay += np.random.uniform(0, 0.5) 
    
    # Add peak/station penalties
    station_delay = cumulative_delay
    if row['is_peak_hour'] == 1:
        station_delay += np.random.uniform(1, 3)
    if row['is_interchange'] == 1:
        station_delay += np.random.uniform(1, 2)
        
    # On-time probability
    if random.random() < 0.15:
        station_delay = 0
        cumulative_delay = 0 # If it catches up, reset propagation
        
    delay_records.append(station_delay)

red_line_df['delay_minutes'] = delay_records
print("Advanced delays generated successfully!")

Generating Advanced Synthetic Delays...
Advanced delays generated successfully!


In [2]:
# Task 1: Data Validation & Cleaning
print(f"Rows before cleaning: {len(red_line_df)}")

# Remove duplicates
red_line_df = red_line_df.drop_duplicates()

# Ensure delay_minutes >= 0 (no negative delays / early arrivals for this model)
red_line_df['delay_minutes'] = red_line_df['delay_minutes'].clip(lower=0)

# Remove unrealistic delays (e.g., > 40 mins)
red_line_df = red_line_df[red_line_df['delay_minutes'] <= 40]

# Drop missing values in crucial columns
red_line_df = red_line_df.dropna(subset=['stop_name', 'delay_minutes'])

print(f"Rows after cleaning: {len(red_line_df)}")

# Task 2: Fix Data Types
red_line_df['hour'] = red_line_df['hour'].astype(int)
red_line_df['day_of_week'] = red_line_df['day_of_week'].astype(int)
red_line_df['is_peak_hour'] = red_line_df['is_peak_hour'].astype(int)
red_line_df['is_weekend'] = red_line_df['is_weekend'].astype(int)
red_line_df['stop_sequence'] = red_line_df['stop_sequence'].astype(int)
red_line_df['direction_id'] = red_line_df['direction_id'].astype(int)
red_line_df['delay_minutes'] = red_line_df['delay_minutes'].round(2).astype(float)

print("\nData Types:")
print(red_line_df[['hour', 'day_of_week', 'is_peak_hour', 'is_weekend', 'delay_minutes']].dtypes)

Rows before cleaning: 30648
Rows after cleaning: 30648

Data Types:
hour               int64
day_of_week        int64
is_peak_hour       int64
is_weekend         int64
delay_minutes    float64
dtype: object


In [3]:
import os

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Task 3: Encode Categorical Features
label_encoder = LabelEncoder()
red_line_df['stop_name_encoded'] = label_encoder.fit_transform(red_line_df['stop_name'])

# Save the encoder for future use (important for Phase 4 Dashboard/Predictions)
joblib.dump(label_encoder, '../models/stop_name_encoder.pkl')
print("Label encoder saved to models/stop_name_encoder.pkl")

Label encoder saved to models/stop_name_encoder.pkl


In [4]:
# Task 6: Feature Selection
features = [
    'stop_name_encoded', 'hour', 'day_of_week', 
    'is_weekend', 'is_peak_hour', 'stop_sequence', 
    'direction_id', 'is_interchange'
]
target = 'delay_minutes'

X = red_line_df[features]
y = red_line_df[target]

# Task 7: Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# Task 8: Feature Scaling
scaler = StandardScaler()

# Fit on training data ONLY, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easy saving/viewing
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=features)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=features)

# Save the scaler
joblib.dump(scaler, '../models/feature_scaler.pkl')
print("Scaler saved to models/feature_scaler.pkl")

Training data shape: (24518, 8)
Testing data shape: (6130, 8)
Scaler saved to models/feature_scaler.pkl


In [5]:
# Task 9: Save Clean Dataset
# Recombine the processed X and y to save a master clean dataset
final_ml_df = X.copy()
final_ml_df['delay_minutes'] = y

# Save to processed data folder
final_ml_df.to_csv('../data/processed/red_line_ml_final.csv', index=False)
print("Final dataset saved to data/processed/red_line_ml_final.csv")

print("\n--- ALL PRE-PHASE 3 TASKS COMPLETED SUCCESSFULLY! ---")
final_ml_df.head()

Final dataset saved to data/processed/red_line_ml_final.csv

--- ALL PRE-PHASE 3 TASKS COMPLETED SUCCESSFULLY! ---


,stop_name_encoded,hour,day_of_week,is_weekend,is_peak_hour,stop_sequence,direction_id,is_interchange,delay_minutes
571,5,6,2,0,0,1,0,0,0.94
572,4,6,4,0,0,2,0,0,1.31
573,26,6,5,1,0,3,0,0,1.61
574,14,6,5,1,0,4,0,0,1.69
575,9,6,5,1,0,1,0,0,0.20
